# 02. 수령 자료 점검, 대상자 특성, 신호 구조 확인

연구기획서 8절의 점검 항목 ②(기반군에서 해마 z와 임상 변수의 연관), ③(추적군 변화율과 기저 변수의 연관)을 수행하고, 결과보고서 표 1(대상자 특성)과 기본 그림을 만든다. 유전체 lead 변이 178개가 질환 변수와 뇌 체적에 어떻게 연결되어 있는지도 확인한다(분석 2의 사전 점검).

입력: `05_분석결과/derived/`의 노트북 01 산출물. 출력 셀에는 집계값만 남긴다.

In [1]:
import json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.model_selection import cross_val_predict, StratifiedKFold, KFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, r2_score
warnings.filterwarnings("ignore")
import logging; logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
import matplotlib.font_manager as fm
_avail = {f.name for f in fm.fontManager.ttflist}
_pref = [f for f in ["AppleGothic", "Malgun Gothic", "NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"] if f in _avail]
plt.rcParams["font.family"] = _pref + ["DejaVu Sans"]; plt.rcParams["axes.unicode_minus"] = False
print("그림 글꼴:", _pref[0] if _pref else "한글 글꼴 없음(영문 대체)")

ROOT = Path.cwd()
while not (ROOT / "02_Data").exists() and ROOT != ROOT.parent: ROOT = ROOT.parent
DERIVED, FIG, TAB = ROOT / "05_분석결과" / "derived", ROOT / "05_분석결과" / "figures", ROOT / "05_분석결과" / "tables"
TAB.mkdir(parents=True, exist_ok=True)
B = pd.read_csv(DERIVED / "base_analysis.csv"); F1 = pd.read_csv(DERIVED / "follow_baseline.csv"); L = pd.read_csv(DERIVED / "follow_longitudinal.csv")
info = json.load(open(DERIVED / "normative_model.json", encoding="utf-8"))
C0, C1, C2 = info["blocks"]["C0"], info["blocks"]["C1"], info["blocks"]["C2"]
DZ = ["htn", "dm", "lip"]
dev, ev = B[B.split == "dev"], B[B.split == "eval"]
print("개발군", len(dev), "| 평가군", len(ev), "| 추적기반", len(F1), "| 추적 종단", len(L))

그림 글꼴: Noto Sans CJK JP
개발군 8000 | 평가군 2000 | 추적기반 1000 | 추적 종단 698


## 1. 표 1 대상자 특성 (개발군, 평가군, 추적기반)

In [2]:
def describe(df, name):
    r = {"집단": name, "n": len(df),
         "연령 평균(SD)": f"{df.age.mean():.1f} ({df.age.std():.1f})", "남성 %": f"{100*df.male.mean():.1f}",
         "고혈압 %": f"{100*df.htn.mean():.1f}", "당뇨 %": f"{100*df.dm.mean():.1f}", "고지혈증 %": f"{100*df.lip.mean():.1f}",
         "BMI 평균(SD)": f"{df.bmi.mean():.1f} ({df.bmi.std():.1f})", "수축기혈압 평균(SD)": f"{df.sbp.mean():.1f} ({df.sbp.std():.1f})",
         "공복혈당 평균(SD)": f"{df.glu.mean():.1f} ({df.glu.std():.1f})", "HbA1c 결측 %": f"{100*df.hba1c.isna().mean():.1f}",
         "현재흡연 %": f"{100*df.smoke_cur.mean():.1f}", "현재음주 %": f"{100*df.drink_cur.mean():.1f}",
         "해마 비율 ×1000 평균(SD)": f"{1000*df.hip_ratio.mean():.2f} ({1000*df.hip_ratio.std():.2f})",
         "해마 z 평균(SD)": f"{df.z_hip.mean():.2f} ({df.z_hip.std():.2f})", "위축 양성 %": f"{100*df.atrophy_pos.mean():.1f}"}
    return r
T1 = pd.DataFrame([describe(dev, "개발군"), describe(ev, "평가군"), describe(F1, "추적기반")]).set_index("집단").T
T1.to_csv(TAB / "table1_characteristics.csv", encoding="utf-8-sig")
print(T1.to_string())

집단                           개발군           평가군          추적기반
n                           8000          2000          1000
연령 평균(SD)             53.7 (8.7)    53.7 (8.7)    54.3 (9.0)
남성 %                        34.6          34.6          44.2
고혈압 %                       20.0          20.3          20.7
당뇨 %                         7.2           6.9           7.0
고지혈증 %                       8.0           7.8           5.2
BMI 평균(SD)            24.0 (3.0)    24.1 (3.0)    24.7 (3.1)
수축기혈압 평균(SD)        123.2 (15.9)  122.8 (16.1)  123.4 (17.8)
공복혈당 평균(SD)          95.5 (21.6)   94.4 (18.6)   90.3 (20.9)
HbA1c 결측 %                  62.3          62.7          31.8
현재흡연 %                      13.1          11.9          19.4
현재음주 %                      44.3          46.4          46.6
해마 비율 ×1000 평균(SD)   4.94 (0.71)   4.94 (0.69)   4.88 (0.68)
해마 z 평균(SD)          0.01 (1.00)   0.00 (0.98)  -0.01 (0.97)
위축 양성 %                     15.2          15.3          14.6


## 2. 점검 ②: 기반군에서 임상 변수가 해마 z와 위축 양성을 얼마나 설명하는가

개발군 5분할 교차검증(규제 로지스틱과 릿지). 이 값은 이후 정식 모델 비교의 대략적인 상한을 미리 보여 준다.

In [3]:
def cv_auc(df, cols, y="atrophy_pos", C=1.0):
    pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=3000, C=C))
    p = cross_val_predict(pipe, df[cols], df[y], cv=StratifiedKFold(5, shuffle=True, random_state=1), method="predict_proba")[:, 1]
    return roc_auc_score(df[y], p)
def cv_r2(df, cols, y="z_hip"):
    pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 20)))
    p = cross_val_predict(pipe, df[cols], df[y], cv=KFold(5, shuffle=True, random_state=1))
    return r2_score(df[y], p)

blocks = {"M0 연령, 성별": C0, "질환 진단력만": DZ, "M1 (C0+C1)": C0 + C1, "M1에서 진단력 제외": C0 + [c for c in C1 if c not in DZ], "M2 (C0+C1+C2)": C0 + C1 + C2}
chk2 = pd.DataFrame([{"블록": k, "AUROC(위축 양성)": round(cv_auc(dev, v), 3), "R²(z_hip)": round(cv_r2(dev, v), 3)} for k, v in blocks.items()])
chk2.to_csv(TAB / "check2_clinical_vs_z.csv", index=False, encoding="utf-8-sig")
print(chk2.to_string(index=False))
print("\n질환별 해마 z 평균(개발군):")
for v in DZ:
    a, b = dev.loc[dev[v] == 1, "z_hip"], dev.loc[dev[v] == 0, "z_hip"]
    t = stats.ttest_ind(a, b, equal_var=False)
    print(f"  {v}=1 (n={len(a)}): {a.mean():.3f}  vs  {v}=0: {b.mean():.3f}   차이 {a.mean()-b.mean():.3f}, p={t.pvalue:.1e}")
print("연속 변수와 z_hip의 상관(개발군, |r| 상위 8):")
cors = dev[C1 + C2 + ["z_hip"]].corr()["z_hip"].drop("z_hip").sort_values(key=np.abs, ascending=False).head(8)
print(cors.round(3).to_string())

           블록  AUROC(위축 양성)  R²(z_hip)
    M0 연령, 성별         0.533     -0.000
      질환 진단력만         0.599      0.050
   M1 (C0+C1)         0.628      0.051
  M1에서 진단력 제외         0.548      0.007
M2 (C0+C1+C2)         0.626      0.050

질환별 해마 z 평균(개발군):
  htn=1 (n=1600): -0.406  vs  htn=0: 0.108   차이 -0.514, p=5.5e-75
  dm=1 (n=575): -0.430  vs  dm=0: 0.040   차이 -0.470, p=9.0e-29
  lip=1 (n=637): -0.138  vs  lip=0: 0.018   차이 -0.156, p=2.7e-04
연속 변수와 z_hip의 상관(개발군, |r| 상위 8):
htn     -0.206
dm      -0.122
hba1c   -0.078
bmi     -0.071
sbp     -0.069
waist   -0.065
dbp     -0.055
glu     -0.055


## 3. 점검 ③: 추적군에서 기저 정보와 이후 변화의 관계

세 가지를 본다. (a) 단면 z가 시간이 지나도 유지되는가(추적 안정성) (b) 기저 임상 정보가 연간 변화율과 가속 위축을 설명하는가 (c) 기저 임상 정보가 10.8년 뒤의 위축 상태(수준)를 설명하는가. 변화율 분석은 평균 회귀의 영향을 받으므로 (a)와 (c)를 함께 해석한다.

In [4]:
L["atrophy5"] = (L.z_hip_5 <= -1).astype(int)
r15 = L[["z_hip", "z_hip_5"]].corr().iloc[0, 1]
print("(a) 추적 안정성: corr(z_hip 기저, z_hip 4차) = %.3f | 해마 원체적 상관 %.3f" % (r15, L[["BV_subctx_hippocampus", "BV_subctx_hippocampus_5"]].corr().iloc[0, 1]))
print("    z_hip 4차 평균/SD = %.3f/%.3f (기저 %.3f/%.3f) → 4차 시점 z 분포가 0 쪽으로 압축됨" % (L.z_hip_5.mean(), L.z_hip_5.std(), L.z_hip.mean(), L.z_hip.std()))
tr = pd.crosstab(L.atrophy_pos, L.atrophy5, normalize="index") * 100
print("    기저 위축 양성 중 4차에도 양성: %.1f%% | 기저 음성 중 4차 양성: %.1f%% | 4차 양성 전체 %.1f%%" % (tr.loc[1, 1], tr.loc[0, 1], 100 * L.atrophy5.mean()))
print("    corr(기저 z_hip, 연간 변화율) = %.3f  (음수면 기저에 작을수록 이후 감소가 느림 = 평균 회귀 방향)" % L[["z_hip", "rate_hip"]].corr().iloc[0, 1])

print("\n(b) 기저 정보 → 연간 변화율과 가속 위축 (5분할 CV):")
def cvs(cols):
    return round(cv_r2(L, cols, "rate_hip"), 3), round(cv_auc(L, cols, "accel_atrophy"), 3)
rows = []
for k, v in {"M0": C0, "M1": C0 + C1, "M2": C0 + C1 + C2, "M1 + 기저 z_hip": C0 + C1 + ["z_hip"]}.items():
    r2v, aucv = cvs(v); rows.append({"블록": k, "R²(연간 변화율)": r2v, "AUROC(가속 위축)": aucv})
print(pd.DataFrame(rows).to_string(index=False))
for v in ["htn", "dm"]:
    a, b = L[L[v] == 1], L[L[v] == 0]
    print(f"    {v}=1 (n={len(a)}): 해마 {a.BV_subctx_hippocampus.mean():.3f} → {a.BV_subctx_hippocampus_5.mean():.3f} mL, 변화율 {a.rate_hip.mean():.3f}%/년 | {v}=0: {b.BV_subctx_hippocampus.mean():.3f} → {b.BV_subctx_hippocampus_5.mean():.3f}, {b.rate_hip.mean():.3f}%/년")

print("\n(c) 기저 정보 → 4차추적 시점 위축 상태 (5분할 CV):")
rows = []
for k, v in {"M0": C0, "M1": C0 + C1, "M2": C0 + C1 + C2, "기저 z_hip만": ["z_hip"], "M1 + 기저 z_hip": C0 + C1 + ["z_hip"]}.items():
    rows.append({"블록": k, "AUROC(4차 위축 양성)": round(cv_auc(L, v, "atrophy5"), 3), "R²(z_hip 4차)": round(cv_r2(L, v, "z_hip_5"), 3)})
chk3 = pd.DataFrame(rows); chk3.to_csv(TAB / "check3_follow_level.csv", index=False, encoding="utf-8-sig")
print(chk3.to_string(index=False))
m = smf.ols("z_hip_5 ~ z_hip + age + male + htn + dm + lip", data=L).fit()
print("    ANCOVA z_hip_5 ~ z_hip + 연령 + 성별 + 질환: z_hip 계수 %.3f (p=%.1e), htn %.3f (p=%.3f), dm %.3f (p=%.3f), lip %.3f (p=%.3f)" % (
    m.params["z_hip"], m.pvalues["z_hip"], m.params["htn"], m.pvalues["htn"], m.params["dm"], m.pvalues["dm"], m.params["lip"], m.pvalues["lip"]))

(a) 추적 안정성: corr(z_hip 기저, z_hip 4차) = 0.832 | 해마 원체적 상관 0.903
    z_hip 4차 평균/SD = 0.151/0.739 (기저 0.027/0.961) → 4차 시점 z 분포가 0 쪽으로 압축됨
    기저 위축 양성 중 4차에도 양성: 44.4% | 기저 음성 중 4차 양성: 1.2% | 4차 양성 전체 6.7%
    corr(기저 z_hip, 연간 변화율) = -0.363  (음수면 기저에 작을수록 이후 감소가 느림 = 평균 회귀 방향)

(b) 기저 정보 → 연간 변화율과 가속 위축 (5분할 CV):
           블록  R²(연간 변화율)  AUROC(가속 위축)
           M0       0.001         0.551
           M1       0.161         0.602
           M2       0.147         0.590
M1 + 기저 z_hip       0.242         0.647
    htn=1 (n=127): 해마 6.637 → 6.447 mL, 변화율 -0.242%/년 | htn=0: 7.276 → 6.696, -0.765%/년
    dm=1 (n=39): 해마 6.511 → 6.441 mL, 변화율 -0.072%/년 | dm=0: 7.198 → 6.663, -0.705%/년

(c) 기저 정보 → 4차추적 시점 위축 상태 (5분할 CV):


           블록  AUROC(4차 위축 양성)  R²(z_hip 4차)
           M0            0.660         0.009
           M1            0.616        -0.007
           M2            0.585        -0.009
    기저 z_hip만            0.955         0.690
M1 + 기저 z_hip            0.939         0.726
    ANCOVA z_hip_5 ~ z_hip + 연령 + 성별 + 질환: z_hip 계수 0.669 (p=9.6e-202), htn 0.306 (p=0.000), dm 0.283 (p=0.000), lip 0.083 (p=0.317)


## 4. 유전체 신호 구조 점검 (분석 2의 사전 확인)

lead 변이 178개의 유전형을 PLINK bed에서 직접 읽어(외부 도구 불필요) 질환 변수, 대사 지표, 뇌 체적 z와의 연관을 본다.
합성 유전체가 "질환을 거쳐서만" 뇌 체적에 연결되는지, 직접 연결도 있는지 확인한다.

In [5]:
G = ROOT / "02_Data" / "유전체_합성데이터"
fam = pd.read_csv(G / "synthetic_kchip_chr1.fam", sep=r"\s+", header=None, names=["FID", "IID", "PAT", "MAT", "SEX", "PHENO"])
N = len(fam); bps = (N + 3) // 4
Bi = B.set_index("T_ID").loc[fam.IID.values]
LUT = np.array([2, np.nan, 1, 0], dtype=float)  # PLINK 2비트: 00=A1/A1(2), 01=결측, 10=이형(1), 11=A2/A2(0)
def read_snps(chrom, idxs):
    with open(G / f"synthetic_kchip_chr{chrom}.bed", "rb") as f:
        assert f.read(3) == b"\x6c\x1b\x01"
        out = []
        for i in idxs:
            f.seek(3 + i * bps); buf = np.frombuffer(f.read(bps), dtype=np.uint8)
            bits = np.unpackbits(buf, bitorder="little").reshape(-1, 2); out.append(LUT[bits[:, 0] + 2 * bits[:, 1]][:N])
    return np.array(out)

phen = {k: Bi[k].values.astype(float) for k in ["htn", "dm", "lip", "sbp", "glu", "ldl", "bmi", "tg", "hdl", "z_hip", "z_wb", "z_vent"]}
rows, mats, names = [], [], []
for c in range(1, 23):
    bim = pd.read_csv(G / f"synthetic_kchip_chr{c}.bim", sep=r"\s+", header=None, names=["CHR", "SNP", "CM", "BP", "A1", "A2"])
    idx = bim.index[bim.SNP.str.match(r"^rs\d+$")].tolist()
    if not idx: continue
    D = read_snps(c, idx); mats.append(D); names += bim.SNP[idx].tolist()
    for k, i in enumerate(idx):
        g = D[k]; r = {"chr": c, "snp": bim.SNP[i], "maf": round(np.nanmean(g) / 2, 3)}
        for ph, y in phen.items():
            ok = ~np.isnan(g) & ~np.isnan(y); lr = stats.linregress(g[ok], y[ok]); r["p_" + ph] = lr.pvalue; r["beta_" + ph] = lr.slope
        rows.append(r)
R = pd.DataFrame(rows); R.to_csv(DERIVED / "lead_snp_scan.csv", index=False)
Gm = np.vstack(mats).T; Gm = np.where(np.isnan(Gm), np.nanmean(Gm, axis=0), Gm)
pd.DataFrame(Gm, columns=names, index=fam.IID.values).rename_axis("T_ID").to_csv(DERIVED / "lead_dosage.csv")
thr = 0.05 / len(R)
print("lead 변이", len(R), "개 | Bonferroni p <", f"{thr:.1e}")
print("유의 lead 수 —", ", ".join(f"{ph}: {int((R['p_'+ph] < thr).sum())}" for ph in phen))
print("해마 z와 연관 상위 5:"); print(R.sort_values("p_z_hip").head(5)[["chr", "snp", "maf", "beta_z_hip", "p_z_hip", "p_htn", "p_dm", "p_lip"]].to_string(index=False))
print("APOE 및 뇌 관련 변이:"); print(R[R.snp.isin(["rs429358", "rs7412", "rs2075650", "rs405509", "rs769446", "rs6265"])][["snp", "maf", "p_z_hip", "p_z_wb", "p_htn", "p_dm", "p_lip"]].to_string(index=False))

devm = (Bi.split.values == "dev")
def lead_cv_auc(y, C=0.01):
    ok = ~np.isnan(y) & devm
    p = cross_val_predict(LogisticRegression(max_iter=3000, C=C), Gm[ok], y[ok].astype(int), cv=StratifiedKFold(5, shuffle=True, random_state=1), method="predict_proba")[:, 1]
    return roc_auc_score(y[ok].astype(int), p)
p = cross_val_predict(RidgeCV(alphas=np.logspace(0, 5, 20)), Gm[devm], phen["z_hip"][devm], cv=KFold(5, shuffle=True, random_state=1))
print("\n개발군 5분할 CV, lead 변이 178개만으로: z_hip R² = %.4f | 위축 양성 AUROC = %.3f" % (r2_score(phen["z_hip"][devm], p), lead_cv_auc(Bi.atrophy_pos.values.astype(float))))
print("                                   고혈압 AUROC = %.3f | 당뇨 AUROC = %.3f | 고지혈증 AUROC = %.3f" % (lead_cv_auc(phen["htn"]), lead_cv_auc(phen["dm"]), lead_cv_auc(phen["lip"])))

lead 변이 178 개 | Bonferroni p < 2.8e-04
유의 lead 수 — htn: 20, dm: 22, lip: 24, sbp: 11, glu: 11, ldl: 0, bmi: 2, tg: 0, hdl: 0, z_hip: 1, z_wb: 1, z_vent: 1
해마 z와 연관 상위 5:
 chr        snp   maf  beta_z_hip  p_z_hip        p_htn     p_dm    p_lip
  10  rs1530440 0.221   -0.069888 0.000036 1.800993e-21 0.005635 0.157408
  11   rs381815 0.263   -0.054362 0.000725 1.648977e-34 0.099782 0.034194
   3  rs3774372 0.171   -0.056705 0.002227 1.593766e-44 0.128919 0.512850
   4 rs16998073 0.296   -0.045251 0.003515 1.518256e-30 0.057320 0.018871
   8  rs1799998 0.394   -0.041104 0.004295 9.334517e-25 0.141509 0.453452
APOE 및 뇌 관련 변이:
      snp   maf  p_z_hip   p_z_wb    p_htn     p_dm        p_lip
   rs6265 0.118 0.590118 0.828886 0.373274 0.869044 4.253161e-01
 rs405509 0.445 0.076939 0.706057 0.255543 0.208857 9.037718e-14
 rs769446 0.116 0.476678 0.138884 0.061084 0.747042 2.589086e-33
   rs7412 0.089 0.836609 0.708107 0.035761 0.446889 4.170579e-23
 rs429358 0.091 0.813794 0.497640 0.803226 0.


개발군 5분할 CV, lead 변이 178개만으로: z_hip R² = 0.0025 | 위축 양성 AUROC = 0.545


                                   고혈압 AUROC = 0.801 | 당뇨 AUROC = 0.810 | 고지혈증 AUROC = 0.820


## 5. 그림 (05_분석결과/figures)

In [6]:
import pickle
norm = pickle.load(open(DERIVED / "normative_models.pkl", "rb"))["hip_ratio"]
ages = np.arange(40, 87)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
samp = dev.sample(min(3000, len(dev)), random_state=1)
ax[0].scatter(samp.age + np.random.default_rng(1).uniform(-0.4, 0.4, len(samp)), 1000 * samp.hip_ratio, s=4, alpha=0.25, color="#7a8aa0")
for male, lab, col in [(1, "남성", "#1f5fa8"), (0, "여성", "#c0392b")]:
    pred = norm.predict(pd.DataFrame({"age": ages, "male": male}))
    sd = info["normative"]["hip_ratio"]["resid_sd"]
    ax[0].plot(ages, 1000 * pred, color=col, lw=2, label=f"{lab} 기대치"); ax[0].plot(ages, 1000 * (pred - sd), color=col, lw=1, ls="--")
ax[0].set_xlabel("연령"); ax[0].set_ylabel("해마 비율 ×1000 (해마/ICV)"); ax[0].set_title("개발군: 연령 및 성별 기대치 곡선과 z = −1 경계(점선)"); ax[0].legend()
for v, col in [("htn", "#1f5fa8"), ("dm", "#c0392b")]:
    ax[1].hist(dev.loc[dev[v] == 1, "z_hip"], bins=40, density=True, alpha=0.45, color=col, label=f"{v}=1")
ax[1].hist(dev.loc[(dev.htn == 0) & (dev.dm == 0), "z_hip"], bins=40, density=True, alpha=0.35, color="#7a8aa0", label="고혈압, 당뇨 없음")
ax[1].axvline(-1, color="k", ls="--", lw=1); ax[1].set_xlabel("해마 z-점수"); ax[1].set_title("질환 유무별 z 분포(개발군)"); ax[1].legend()
plt.tight_layout(); plt.savefig(FIG / "fig_norm_curve_and_z_by_disease.png", dpi=150); plt.close()

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].scatter(L.z_hip, L.z_hip_5, s=8, alpha=0.5, color="#1f5fa8"); ax[0].plot([-4, 3], [-4, 3], color="k", lw=1, ls="--")
ax[0].set_xlabel("기저 해마 z"); ax[0].set_ylabel("4차추적 해마 z"); ax[0].set_title(f"추적 안정성 r = {r15:.2f}")
dec = pd.qcut(L.z_hip, 10, labels=False)
g = L.groupby(dec).rate_hip.agg(["mean", "sem"])
ax[1].errorbar(np.arange(1, 11), g["mean"], yerr=1.96 * g["sem"], fmt="o-", color="#c0392b")
ax[1].set_xlabel("기저 해마 z 십분위(1 = 가장 작음)"); ax[1].set_ylabel("해마 연간 변화율 (%/년)"); ax[1].set_title("기저 z 십분위별 연간 변화율(평균 회귀 확인)")
plt.tight_layout(); plt.savefig(FIG / "fig_follow_tracking_and_rtm.png", dpi=150); plt.close()

fig, ax = plt.subplots(figsize=(11, 3.8))
x = np.arange(len(R))
for ph, col, off in [("htn", "#1f5fa8", -0.25), ("dm", "#c0392b", 0), ("z_hip", "#2e8b57", 0.25)]:
    ax.bar(x + off, -np.log10(R["p_" + ph].clip(lower=1e-50)), width=0.25, color=col, label={"htn": "고혈압", "dm": "당뇨", "z_hip": "해마 z"}[ph])
ax.axhline(-np.log10(thr), color="k", ls="--", lw=1); ax.set_xlabel("lead 변이 (염색체 순)"); ax.set_ylabel("−log10 p"); ax.set_ylim(0, 45)
ax.set_title("lead 변이 178개의 질환 및 해마 z 연관"); ax.legend()
plt.tight_layout(); plt.savefig(FIG / "fig_lead_snp_scan.png", dpi=150); plt.close()
print("그림 저장:", sorted(p.name for p in FIG.iterdir()))

그림 저장: ['fig_follow_tracking_and_rtm.png', 'fig_lead_snp_scan.png', 'fig_norm_curve_and_z_by_disease.png']
